# Categorical predictors and estimator inputs

A label can be a pandas category, a string or an integer code. The dtype
and estimator determine how the label is handled. This fixture uses a
nominal category with one predictive level and a numeric noise predictor.
Define the known category vocabulary in the example and reuse the training
schema for held-out rows.

In [1]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from arfs.feature_selection import BoostAGroota, GrootCV, Leshy

rng = np.random.default_rng(42)
tiers = ["bronze", "silver", "gold"]
X = pd.DataFrame(
    {
        "tier": pd.Categorical(rng.choice(tiers, size=360), categories=tiers),
        "noise": rng.normal(size=360),
    }
)
y = np.where(rng.random(len(X)) < 0.08, X["tier"] != "silver", X["tier"] == "silver")
X_train, X_test, y_train, y_test = train_test_split(
    X, y.astype(int), stratify=y, random_state=42
)
X_train.dtypes

tier     category
noise     float64
dtype: object

## LightGBM-backed all-relevant selectors

Pass the category dtype to GrootCV or LightGBM-backed Leshy and
BoostAGroota. ARFS handles category coding and tells LightGBM which
columns are categorical. If raw integer codes represent unordered labels,
cast them to category before selection. A standalone LightGBM sklearn
estimator can use this unordered pandas category directly. Reuse its
training categories when serving new rows.

In [2]:
tree = LGBMClassifier(n_estimators=35, n_jobs=1, random_state=42, verbosity=-1)
selectors = {
    "GrootCV": GrootCV(
        objective="binary",
        n_folds=3,
        n_iter=1,
        n_jobs=1,
        random_state=42,
        lgbm_params={"num_iterations": 35},
    ),
    "Leshy": Leshy(tree, n_estimators=35, max_iter=15, random_state=42),
    "BoostAGroota": BoostAGroota(tree, iters=3, max_rounds=3, random_state=42),
}
pd.Series(
    {
        name: selector.fit(X_train, y_train).selected_features_.tolist()
        for name, selector in selectors.items()
    },
    name="Selected original columns",
).to_frame()

,Selected original columns
GrootCV,"[noise, tier]"
Leshy,[tier]
BoostAGroota,"[tier, noise]"


In [3]:
selected = selectors["GrootCV"]
train_selected = selected.transform(X_train)
test_selected = selected.transform(X_test)
assert train_selected["tier"].dtype == test_selected["tier"].dtype
tree.fit(train_selected, y_train)
tree.predict_proba(test_selected)[:3]

array([[0.93505101, 0.06494899],
       [0.98157912, 0.01842088],
       [0.87107589, 0.12892411]])

## An estimator that requires a numeric design

Logistic regression needs numeric inputs. One-hot encode the nominal
column so each level can have its own effect. Fit the encoder on the
training rows and reuse it on the held-out rows. If you run Lasso after
this encoding, selection operates on dummy columns.

In [4]:
design = ColumnTransformer(
    [("tier", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["tier"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")
train_encoded = design.fit_transform(X_train)
test_encoded = design.transform(X_test)
linear = LogisticRegression().fit(train_encoded, y_train)
pd.DataFrame({"coefficient": linear.coef_[0]}, index=test_encoded.columns)

,coefficient
tier_bronze,-1.350508
tier_gold,-1.365334
tier_silver,2.715242
noise,-0.026385
